# 01 EDA

Explore the ISIC 2024 training data: image counts, patient counts, class balance, missing values, image sizes, and random examples.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

_ = plt
_ = Image

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

DATA_DIR = Path('data')
TRAIN_METADATA_PATH = DATA_DIR / 'train-metadata.csv'
TRAIN_IMAGE_DIR = DATA_DIR / 'train-image'

print('Data directory:', DATA_DIR.resolve())

In [ ]:
if TRAIN_METADATA_PATH.exists():
    train_df = pd.read_csv(TRAIN_METADATA_PATH)
    print(f'Loaded {len(train_df):,} rows from {TRAIN_METADATA_PATH.name}')
else:
    train_df = pd.DataFrame()
    print(f'Missing {TRAIN_METADATA_PATH}. Place the Kaggle train metadata there before running the notebook.')

image_files = []
if TRAIN_IMAGE_DIR.exists():
    image_files = [path for path in TRAIN_IMAGE_DIR.rglob('*') if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    print(f'Found {len(image_files):,} image files under {TRAIN_IMAGE_DIR}')
else:
    print(f'Missing image directory: {TRAIN_IMAGE_DIR}')

In [ ]:
if train_df.empty:
    print('EDA summary is unavailable until train-metadata.csv is added.')
else:
    display(train_df.head())
    print('Number of images:', len(train_df))
    if 'patient_id' in train_df.columns:
        print('Number of patients:', train_df['patient_id'].nunique())
    if 'target' in train_df.columns:
        print('Positive samples:', int(train_df['target'].sum()))
        print('Negative samples:', int((train_df['target'] == 0).sum()))
        train_df['target'].value_counts(dropna=False).plot(kind='bar', title='Target distribution')
        plt.show()
    print('Missing values per column:')
    display(train_df.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
if image_files:
    sample_images = image_files[:5] if len(image_files) < 5 else pd.Series(image_files).sample(5, random_state=42).tolist()
    figures = plt.figure(figsize=(15, 8))
    for idx, image_path in enumerate(sample_images, start=1):
        with Image.open(image_path) as image:
            width, height = image.size
            axis = figures.add_subplot(1, len(sample_images), idx)
            axis.imshow(image)
            axis.set_title(f'{image_path.name}\n{width}x{height}')
            axis.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No image files found to display.')

## Deliverable

Use this notebook to confirm dataset size, class balance, metadata quality, and example image quality before moving to preprocessing and model training.